# Battery Air Cooling Simulator - Demo Notebook

This notebook demonstrates the key features of the battery air cooling simulator:

1. **Single Configuration Simulation**
2. **Parameter Sweep Analysis**
3. **ROM vs CFD Comparison**
4. **Pareto Front Analysis**
5. **Configuration Ranking**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from battery_aircooling.config_schema import SimulationConfig
from battery_aircooling.physics_rom import ROMSolver
from battery_aircooling.physics_post import (
    calculate_metrics,
    metrics_to_dict,
    summarize_results,
    calculate_pareto_front,
    compare_configurations,
)
from battery_aircooling.visualize import (
    plot_temperature_distribution,
    plot_channel_flow_distribution,
    plot_sweep_results,
    plot_pareto_front,
    plot_spider_chart,
)

# Enable inline plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Single Configuration Simulation

Load a configuration and run a single simulation to understand the baseline performance.

In [ ]:
# Load configuration
config = SimulationConfig.from_yaml("study_rib_sweep.yaml")

# Create solver and run
solver = ROMSolver(config)
results = solver.solve()

# Calculate metrics
metrics = calculate_metrics(results, config.metrics)

# Print summary
print(summarize_results(metrics))

In [ ]:
# Visualize temperature distribution
plot_temperature_distribution(results, show=True)

In [ ]:
# Visualize channel flow distribution
plot_channel_flow_distribution(results.channels, show=True)

## 2. Parameter Sweep: Rib Height

Investigate how rib height affects cooling performance.

In [ ]:
# Define sweep parameters
rib_heights = np.linspace(0.001, 0.008, 15)  # 1mm to 8mm

# Run sweep
metrics_list = []

for height in rib_heights:
    config_copy = config.model_copy(deep=True)
    config_copy.ribs.height = height
    
    solver = ROMSolver(config_copy)
    result = solver.solve()
    
    metrics = calculate_metrics(result, config.metrics)
    metrics_list.append(metrics)

# Plot results
plot_sweep_results(
    "Rib Height [mm]",
    rib_heights * 1000,  # Convert to mm
    metrics_list,
    show=True
)

## 3. Parameter Sweep: Rib Pitch

Investigate the effect of rib spacing.

In [ ]:
# Define sweep parameters
rib_pitches = np.linspace(0.005, 0.040, 15)  # 5mm to 40mm

# Run sweep
metrics_pitch_list = []

for pitch in rib_pitches:
    config_copy = config.model_copy(deep=True)
    config_copy.ribs.pitch = pitch
    
    solver = ROMSolver(config_copy)
    result = solver.solve()
    
    metrics = calculate_metrics(result, config.metrics)
    metrics_pitch_list.append(metrics)

# Plot results
plot_sweep_results(
    "Rib Pitch [mm]",
    rib_pitches * 1000,
    metrics_pitch_list,
    show=True
)

## 4. Multi-Configuration Comparison

Compare different rib geometries.

In [ ]:
# Define configurations to compare
rib_types = ["rect", "tri", "semi", "dimple"]
config_names = ["Rectangular", "Triangular", "Semicircular", "Dimple"]

comparison_metrics = []

for rib_type in rib_types:
    config_copy = config.model_copy(deep=True)
    config_copy.ribs.type = rib_type
    
    solver = ROMSolver(config_copy)
    result = solver.solve()
    
    metrics = calculate_metrics(result, config.metrics)
    comparison_metrics.append(metrics)

# Compare configurations
comparison = compare_configurations(comparison_metrics, config_names)

# Display results
print("\nBest Configurations:")
for criterion, best in comparison["best"].items():
    print(f"  {criterion}: {best}")

In [ ]:
# Spider chart comparison
plot_spider_chart(comparison_metrics, config_names, show=True)

## 5. Pareto Front Analysis

Identify trade-offs between thermal performance and pumping power.

In [ ]:
# Combine all metrics from sweeps
all_metrics = metrics_list + metrics_pitch_list + comparison_metrics

# Find Pareto front
pareto_indices = calculate_pareto_front(all_metrics, "T_max", "P_pump")

print(f"\nFound {len(pareto_indices)} Pareto-optimal configurations")

# Plot Pareto front
plot_pareto_front(
    all_metrics,
    pareto_indices=pareto_indices,
    objective1="T_max",
    objective2="P_pump",
    show=True
)

## 6. Export Results

Save results for further analysis.

In [ ]:
# Create DataFrame with all results
results_data = [metrics_to_dict(m) for m in all_metrics]
df = pd.DataFrame(results_data)

# Display summary statistics
print("\nSummary Statistics:")
print(df[["T_max_C", "dT_pack_K", "P_pump_W", "JF_factor"]].describe())

# Save to CSV
df.to_csv("sweep_analysis_results.csv", index=False)
print("\n✓ Results saved to sweep_analysis_results.csv")

## Conclusions

Key findings from this analysis:

1. **Rib Height**: Higher ribs improve heat transfer but increase pressure drop
2. **Rib Pitch**: Optimal pitch balances heat transfer enhancement and pressure loss
3. **Rib Shape**: Different geometries offer varying thermal-hydraulic performance
4. **Trade-offs**: Clear Pareto front between cooling effectiveness and pumping power

The optimal configuration depends on the relative importance of:
- Maximum temperature limit
- Temperature uniformity
- Energy consumption (pumping power)
- System constraints (pressure drop limit)